# Faster-RCNN via DaisyTuner on Q.ANT Gen2 NPU

In [ ]:
import torch
import torchvision.models as models
import time
import matplotlib.pyplot as plt
import PIL.Image as Image
import requests

import docc.torch
from docc.qant import register_docc_plugin

register_docc_plugin()

In [ ]:
# Load a pre-trained Faster R-CNN model
weights = models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = models.detection.fasterrcnn_resnet50_fpn(weights=weights)
model.eval()
transforms = weights.transforms()


# COCO class labels mapping
labels = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant',
    'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A',
    'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana',
    'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut',
    'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A',
    'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'microwave',
    'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book', 'clock', 'vase',
    'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]


In [ ]:
# Compile only the backbone with docc, rest of model runs eagerly
print("\n--- Compile Times ---")
start = time.perf_counter()
program_ref = torch.compile(model)
torch_compile_time = time.perf_counter() - start
print(f"Torch compile: {torch_compile_time:.4f} s")


start = time.perf_counter()
program_backbone = torch.compile(model.backbone, backend="docc", options={"target": "qant", "category": "server"})
docc_compile_time = time.perf_counter() - start
print(f"Docc compile: {docc_compile_time:.4f} s")

# Replace the backbone with compiled version
model.backbone = program_backbone


In [ ]:
# select a image
url = "https://github.com/pytorch/hub/raw/master/images/dog.jpg"  # dog (Samoyed)

image = Image.open(requests.get(url, stream=True).raw)


input_tensor = transforms(image)
input_batch = input_tensor.unsqueeze(0)  # create a mini-batch as expected by the model

_ = plt.imshow(image)
plt.title("Input Image")
plt.axis("off")

In [ ]:
# Force dynamo (inference) backend
with torch.no_grad():
    print("--- Execution Times ---")
    start = time.perf_counter()
    res = model(input_batch)
    docc_exec_time = time.perf_counter() - start
    print(f"Docc first execution: {docc_exec_time:.4f} s")

    start = time.perf_counter()
    ref = program_ref(input_batch)
    torch_exec_time = time.perf_counter() - start
    print(f"Torch first execution: {torch_exec_time:.4f} s")

    start = time.perf_counter()
    res = model(input_batch)
    docc_exec_time = time.perf_counter() - start
    print(f"Docc 2nd execution: {docc_exec_time:.4f} s")

    start = time.perf_counter()
    ref = program_ref(input_batch)
    torch_exec_time = time.perf_counter() - start
    print(f"Torch 2nd execution: {torch_exec_time:.4f} s")


# Draw the predicted bounding boxes
def draw_boxes(image, boxes, classes ,labels, scores, threshold=0.4, title="Predicted Bounding Boxes"):
    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    ax = plt.gca()

    box_labels = [labels[min(i, len(labels) - 1)] for i in classes]

    for box, label, score in zip(boxes, box_labels, scores):
        if score >= threshold:
            x1, y1, x2, y2 = box
            rect = plt.Rectangle(
                (x1, y1), x2 - x1, y2 - y1, fill=False, color="red", linewidth=2
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 10, f"{label}: {score:.2f}", color="red", fontsize=12)

    plt.axis("off")
    plt.title(title)
    plt.show()


draw_boxes(image, ref[0]["boxes"].cpu().numpy(), ref[0]["labels"].cpu().numpy(), labels, ref[0]["scores"].cpu().numpy(), threshold=0.5, title="Torch Detections")
draw_boxes(image, res[0]["boxes"].cpu().numpy(), res[0]["labels"].cpu().numpy(), labels, res[0]["scores"].cpu().numpy(), threshold=0.5, title="Q.ANT Detections")
